In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

In [ ]:
# Create a DataFrame from Table 5 data
data = {
    "Feature_Count": [200, 400, 600, 800, 1000],
    "ACC": [0.8520, 0.8688, 0.8886, 0.9790, 0.9385],
    "AUC": [0.9712, 0.9785, 0.9830, 0.9857, 0.9861],
    "PRE": [0.8490, 0.8722, 0.8854, 0.9066, 0.9054],
    "SP": [0.8413, 0.8586, 0.8797, 0.9029, 0.9010],
    "SN": [0.8439, 0.8616, 0.8811, 0.9040, 0.9026],
    "F1": [0.8017, 0.8257, 0.8511, 0.8795, 0.8774],
    "MCC": [0.8520, 0.8688, 0.8886, 0.9790, 0.9085]
}

In [ ]:
df = pd.DataFrame(data)

In [ ]:
# Reset matplotlib settings to default
plt.rcParams.update(plt.rcParamsDefault)

# Set Times New Roman font with fallback
plt.rcParams['font.family'] = ['Times New Roman', 'serif']
plt.rcParams['mathtext.fontset'] = 'stix'

# Set global plot style with adjusted font sizes
plt.rcParams['font.size'] = 28
plt.rcParams['axes.labelsize'] = 32
plt.rcParams['axes.titlesize'] = 36
plt.rcParams['xtick.labelsize'] = 26
plt.rcParams['ytick.labelsize'] = 26
plt.rcParams['legend.fontsize'] = 24
plt.rcParams['figure.dpi'] = 1000
plt.rcParams['savefig.dpi'] = 1000
plt.rcParams['figure.facecolor'] = 'white'

In [ ]:
# Color palette for Feature Count
colors = ['#1F77B4', '#FF7F0E', '#D62728', '#2CA02C', '#9467BD']
feature_count_palette = dict(zip([200, 400, 600, 800, 1000], colors))

In [ ]:
# Function to save figures in both PNG and PDF in a dedicated folder
def save_figure(fig, filename):
    output_folder = "Experiment_Output_Figures/Feature_Count"
    if not os.path.exists(output_folder):
        os.makedirs(output_folder)

    png_path = os.path.join(output_folder, f"{filename}.png")
    pdf_path = os.path.join(output_folder, f"{filename}.pdf")

    fig.savefig(png_path, dpi=1000, bbox_inches='tight', facecolor='white', format='png')
    fig.savefig(pdf_path, dpi=1000, bbox_inches='tight', facecolor='white', format='pdf')

    print(f"Saved: {png_path} and {pdf_path}")
    plt.show()
    plt.close(fig)

In [ ]:
# Function to generate the violin plot visualization
def generate_violin_plot():
    metrics = ['ACC', 'AUC', 'PRE', 'SP', 'SN', 'F1', 'MCC']
    melted_df = pd.melt(df, id_vars=['Feature_Count'], value_vars=metrics,
                        var_name='Metric', value_name='Score')

    fig, ax = plt.subplots(figsize=(20, 12))
    sns.violinplot(x='Feature_Count', y='Score', hue='Feature_Count', data=melted_df,
                   palette=colors, inner='box', linewidth=2, ax=ax, legend=False)

    ax.set_title('Distribution of Performance Scores by Feature Count', pad=20)
    ax.set_xlabel('Feature Count', labelpad=15)
    ax.set_ylabel('Score Distribution Across Metrics', labelpad=15)
    ax.set_ylim(0.7, 1.1)

    plt.tight_layout()
    save_figure(fig, "feature_count_violin_plot")

In [ ]:
metrics = ['ACC', 'AUC', 'PRE', 'SP', 'SN', 'F1', 'MCC']

In [ ]:
# 1. Line chart showing trends across feature counts
fig, ax = plt.subplots(figsize=(20, 12))
for metric in metrics:
    ax.plot(df['Feature_Count'], df[metric], marker='o', linewidth=2.5,
            markersize=10, label=metric)

ax.set_title('Performance Metrics Trends Across Feature Counts', pad=20)
ax.set_xlabel('Feature Count', labelpad=15)
ax.set_ylabel('Score', labelpad=15)
ax.set_ylim(0.8, 1.0)
ax.set_xticks(df['Feature_Count'])
ax.legend(title='Metrics', title_fontsize=26, bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
save_figure(fig, "feature_count_metrics_trends")

In [ ]:
# 2. Bar chart comparing all feature counts across metrics
melted_df = pd.melt(df, id_vars=['Feature_Count'], value_vars=metrics,
                    var_name='Metric', value_name='Score')

fig, ax = plt.subplots(figsize=(20, 12))
sns.barplot(x='Metric', y='Score', hue='Feature_Count', data=melted_df,
            palette=colors, ax=ax, edgecolor='none')

ax.set_title('Comparison of Feature Counts across Metrics', pad=20)
ax.set_xlabel('Evaluation Metric', labelpad=15)
ax.set_ylabel('Score', labelpad=15)
ax.set_ylim(0.8, 1.0)
ax.legend(title='Feature Count', title_fontsize=26, bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
save_figure(fig, "feature_count_comparison_grouped_bar")

In [ ]:
# 3. Radar/Spider Chart for each Feature Count
categories = metrics
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(16, 16), subplot_kw=dict(polar=True))
for i, count in enumerate(df['Feature_Count']):
    values = df.loc[i, metrics].values.tolist()
    values += values[:1]
    ax.plot(angles, values, linewidth=3, label=str(count), color=colors[i % len(colors)])
    ax.fill(angles, values, alpha=0.1, color=colors[i % len(colors)])

ax.set_ylim(0.8, 1.0)
plt.xticks(angles[:-1], categories)
ax.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1), title='Feature Count', title_fontsize=26)
plt.title('Feature Count Comparison (Radar Chart)', pad=30, y=1.08)

plt.tight_layout()
save_figure(fig, "feature_count_radar_chart")

In [ ]:
# 4. Heatmap showing all scores
pivot_df = df.set_index('Feature_Count')
fig, ax = plt.subplots(figsize=(18, 10))
heatmap = sns.heatmap(pivot_df, annot=True, fmt=".4f", cmap="YlGnBu",
                      linewidths=0.5, linecolor='white', annot_kws={'size': 24}, ax=ax)

cbar = heatmap.collections[0].colorbar
cbar.set_label('Score', size=28)
cbar.ax.tick_params(labelsize=24)

ax.set_title('Feature Count Performance Heatmap', pad=20)
ax.set_xlabel('Evaluation Metrics', labelpad=15)
ax.set_ylabel('Feature Count', labelpad=15)

plt.tight_layout()
save_figure(fig, "feature_count_heatmap")

In [ ]:
# 5. Individual line charts for each metric
for metric in metrics:
    fig, ax = plt.subplots(figsize=(16, 10))
    ax.plot(df['Feature_Count'], df[metric], marker='o', linewidth=2.5,
            markersize=10, color='#4C72B0')

    for i, count in enumerate(df['Feature_Count']):
        ax.text(count, df[metric].iloc[i] + 0.005, f'{df[metric].iloc[i]:.4f}',
                ha='center', va='bottom', fontsize=24)

    best_idx = df[metric].idxmax()
    best_count = df['Feature_Count'].iloc[best_idx]
    best_value = df[metric].iloc[best_idx]
    ax.scatter(best_count, best_value, s=200, color='red',
               edgecolor='black', zorder=10, label=f'Best: {best_count} ({best_value:.4f})')

    ax.set_title(f'{metric} Scores by Feature Count', pad=20)
    ax.set_xlabel('Feature Count', labelpad=15)
    ax.set_ylabel(f'{metric} Score', labelpad=15)
    ax.set_xticks(df['Feature_Count'])
    ax.set_ylim(0.8, 1.0)
    ax.legend(fontsize=24, bbox_to_anchor=(1.05, 1), loc='upper left')

    plt.tight_layout()
    save_figure(fig, f"feature_count_{metric}_trend")

In [ ]:
# 6. Summary bar chart showing average performance
df['Average'] = df[metrics].mean(axis=1)
sorted_df = df.sort_values(by='Feature_Count')

fig, ax = plt.subplots(figsize=(16, 10))
bars = ax.bar(sorted_df['Feature_Count'].astype(str), sorted_df['Average'],
              color=[colors[i % len(colors)] for i in range(len(sorted_df))], width=0.6)

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2., height + 0.005, f'{height:.4f}',
            ha='center', va='bottom', fontsize=24)

ax.set_title('Average Performance by Feature Count', pad=20)
ax.set_xlabel('Feature Count', labelpad=15)
ax.set_ylabel('Average Score Across All Metrics', labelpad=15)
ax.set_ylim(0.8, 1.0)

plt.tight_layout()
save_figure(fig, "feature_count_average_performance")

In [ ]:
# 7. Best performing feature count for each metric
best_counts = {}
for metric in metrics:
    best_idx = df[metric].idxmax()
    best_counts[metric] = {
        'Best Feature Count': df['Feature_Count'].iloc[best_idx],
        'Best Score': df[metric].iloc[best_idx]
    }

best_df = pd.DataFrame.from_dict(best_counts, orient='index')
print("\nBest Feature Count for Each Metric:")
print(best_df)

best_output_folder = "experiment_output_figures/feature_count"
if not os.path.exists(best_output_folder):
    os.makedirs(best_output_folder)
best_df.to_csv(os.path.join(best_output_folder, "best_feature_counts.csv"))
print(f"Best feature counts saved to {os.path.join(best_output_folder, 'best_feature_counts.csv')}")

In [ ]:
# Execute the visualization functions
print("Generating visualizations for ResNet50 + SHAP feature count analysis...")

In [ ]:
generate_violin_plot()

In [ ]:
print("All visualizations have been generated!")
print("Files are saved in the 'experiment_output_figures/feature_count' folder")